In [1]:
# ============================================================================
# BLOCKS 2-3: TEXT CLEANING & FILTERING - ROMANIAN
# Input: 01_loaded_data_ro.pkl
# Output: 02_cleaned_data_ro.pkl, 03_filtered_data_ro.pkl
# ============================================================================
%run 00_setup_and_config.ipynb

c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\election_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 04:09:34
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [2]:
print('='*70)
print('BLOCKS 2-3: TEXT CLEANING & FILTERING - ROMANIAN')
print('='*70)

if check_checkpoint_exists('ro', '03_filtered_data'):
    df_ro = load_checkpoint('ro', '03_filtered_data')
    print('✓ Skipping to filtered data checkpoint')
elif check_checkpoint_exists('ro', '02_cleaned_data'):
    df_ro = load_checkpoint('ro', '02_cleaned_data')
    print('✓ Loading from cleaned data checkpoint')
else:
    df_ro = load_checkpoint('ro', '01_loaded_data')
    if df_ro is None:
        raise FileNotFoundError('Run 01_data_loading_ro.ipynb first')

initial_rows = len(df_ro)
print(f'\nStarting with: {initial_rows:,} rows')

BLOCKS 2-3: TEXT CLEANING & FILTERING - ROMANIAN
✓ Loading checkpoint: 01_loaded_data_ro.pkl

Starting with: 6,790 rows


In [3]:
# BLOCK 2
print('\n--- BLOCK 2: TEXT CLEANING ---\n')

if 'comment_text' not in df_ro.columns:
    raise KeyError('comment_text column not found')

print('1. Removing rows with null comment_text...')
before = len(df_ro)
df_ro = df_ro.dropna(subset=['comment_text']).copy()
print(f'   Rows removed: {before - len(df_ro):,}')

print('\n2. Cleaning text (URLs, emoji, HTML, mentions)...')
df_ro['clean_text'] = df_ro['comment_text'].apply(clean_social_text)

print('\n3. Removing empty or whitespace-only comments...')
df_ro = remove_empty_texts(df_ro, 'clean_text')

save_checkpoint(df_ro, 'ro', '02_cleaned_data')

# BLOCK 3
print('\n--- BLOCK 3: QUALITY FILTERING ---\n')

print('4. Removing exact duplicate comments...')
df_ro = remove_duplicates(df_ro, subset=['video_id', 'author_name', 'comment_text'])

print('\n5. Computing text statistics...')
df_ro['char_len'] = df_ro['clean_text'].str.len()
df_ro['word_count'] = df_ro['clean_text'].str.split().str.len()

print('\nText Metrics Summary:')
display(df_ro[['char_len', 'word_count']].describe())

save_checkpoint(df_ro, 'ro', '03_filtered_data')
update_pipeline_status('ro', 2, 'completed')
update_pipeline_status('ro', 3, 'completed')

final_rows = len(df_ro)
retention = 100 * final_rows / initial_rows

print('\n' + '='*70)
print('CLEANING SUMMARY - ROMANIAN')
print('='*70)
print(f'  Started with: {initial_rows:,} rows')
print(f'  Ended with:   {final_rows:,} rows')
print(f'  Retention:    {retention:.1f}%')
print(f'  Removed:      {initial_rows - final_rows:,} rows')
print('='*70)
print('✓ BLOCKS 2-3 COMPLETE')
print('='*70)


--- BLOCK 2: TEXT CLEANING ---

1. Removing rows with null comment_text...
   Rows removed: 3

2. Cleaning text (URLs, emoji, HTML, mentions)...

3. Removing empty or whitespace-only comments...
  Removed 162 empty text rows (2.4%)
✓ Checkpoint saved: 02_cleaned_data_ro.pkl

--- BLOCK 3: QUALITY FILTERING ---

4. Removing exact duplicate comments...
  Removed 25 duplicate rows (0.4%)

5. Computing text statistics...

Text Metrics Summary:


,char_len,word_count
count,6600.000000,6600.000000
mean,150.846212,26.839091
std,249.576898,43.898347
min,1.000000,1.000000
25%,35.000000,6.000000
50%,77.000000,14.000000
75%,166.000000,30.000000
max,6007.000000,1016.000000


✓ Checkpoint saved: 03_filtered_data_ro.pkl
✓ Pipeline status updated: ro - Block 2 - completed
✓ Pipeline status updated: ro - Block 3 - completed

CLEANING SUMMARY - ROMANIAN
  Started with: 6,790 rows
  Ended with:   6,600 rows
  Retention:    97.2%
  Removed:      190 rows
✓ BLOCKS 2-3 COMPLETE
